# Extended Zero-Shot Classification with Transformer Pipelines

In notebook 2 we used zero-shot classification to label spam. Here we explore the same pipeline more widely: we classify custom sentences with our own labels, see how the method works internally, run it across more than 100 languages, and try other pipeline tasks such as text generation and sentiment analysis.

## Learning Objectives

At the end of this notebook, you should be able to:

- Apply a zero-shot pipeline to classify custom text with your own candidate labels.
- Explain how natural language inference makes zero-shot classification possible.
- Use a multilingual model to classify text across more than 100 languages.
- Compare different pipeline tasks such as text generation and sentiment analysis.

In [ ]:
from transformers import pipeline

import pandas as pd

In [ ]:
classifier = pipeline(
    "zero-shot-classification", device=0
)  # device=0 uses the first GPU; set device=-1 to run on CPU

## Zero-shot classification on custom text

We pass the pipeline a sequence and a list of candidate labels. By default it assumes exactly one label is correct and returns a score per label, where the scores add up to 1. The examples below show how the choice of labels, and even spelling, changes the result.

In [ ]:
"""
We create a function to display our predictions from the model in a tabular form
"""


def get_predictions_score(prediction):
    pred_labels = prediction["labels"]
    pred_scores = prediction["scores"]
    seq = [prediction["sequence"]]
    return (
        pd.concat(
            [
                pd.DataFrame(seq),
                pd.DataFrame(pred_labels),
                pd.DataFrame(pred_scores),
            ],
            axis=1,
            ignore_index=True,
        )
        .rename(columns={0: "Sequence", 1: "Labels", 2: "Probability"})
        .set_index(["Sequence"])
    )

In [ ]:
sequence = "Amazon is the longest river in the world"
candidate_labels = ["geography", "delivery"]

pred = classifier(sequence, candidate_labels)
get_predictions_score(pred)

What if we change the spelling? Here we change `Amazon` to `amazon`. It makes little difference in this case, but in others it will. Try changing the spelling and adding or removing labels, then watch how the scores move.

In [ ]:
sequence = "amazon is the longest river in the world"
candidate_labels = ["geography", "delivery"]

pred = classifier(sequence, candidate_labels)
get_predictions_score(pred)

The example below shows how well these models use context, even with a slight spelling mistake. Try changing the spelling and observe the results.

In [ ]:
sequence = "are we going to Oktoberfest?"
candidate_labels = [
    "food",
    "Munich",
    "bear",
    "wine",
    "pretzel",
    "sausage",
]  ## What if you change bear (animal) -> beer (drink)

pred = classifier(sequence, candidate_labels)
get_predictions_score(pred)

In [ ]:
sequence = "Who are you voting for in 2020?"
candidate_labels = ["food", "public health", "plants", "fruits", "america"]

pred = classifier(sequence, candidate_labels)
get_predictions_score(pred)

> The predictions are poor here because the labels are not related to the sequence. We can improve this by providing candidate labels that fit the input text.

In [ ]:
## Think about other labels which can improve the predictions
## HINT: Labels related to your text

To allow multiple labels independently, pass `multi_label=True`. Each score falls between 0 and 1, but the scores do not need to sum to 1.

In [ ]:
sequence = "Who are you voting for in 2020?"
candidate_labels = ["politics", "public health", "economics", "elections"]

pred = classifier(sequence, candidate_labels, multi_label=True)
get_predictions_score(pred)

### Sentiment classification

Here's an example of sentiment classification: 

In [ ]:
sequence = "I hated this movie. The acting sucked."
candidate_labels = ["positive", "negative"]

pred = classifier(sequence, candidate_labels)
get_predictions_score(pred)

## How zero-shot classification works

The underlying model is trained on **natural language inference (NLI)**: given two sentences, it decides whether they contradict each other, entail each other, or neither.

This is adapted to zero-shot classification by treating the text we want to classify as one NLI sentence (the premise) and turning each candidate label into the other (the hypothesis). If the model predicts that the premise entails the hypothesis, we read that as a prediction that the label applies.

By default the pipeline turns labels into hypotheses with the template `This example is {class_name}.`. This works well in many settings, but you can customise it. Let us add a harder review to the sentiment example:

In [ ]:
sequences = [
    "I hated this movie. The acting sucked.",
    "This movie didn't quite live up to my high expectations, but overall I still really enjoyed it.",
]
candidate_labels = ["positive", "negative"]

classifier(sequences, candidate_labels)

The second example is a bit harder. Let's see if we can improve the results by using a hypothesis template which is more specific to the setting of review sentiment analysis. Instead of the default, `This example is {}.`, we'll use, `The sentiment of this review is {}.` (where `{}` is replaced with the candidate class name)

In [ ]:
sequences = [
    "I hated this movie. The acting sucked.",
    "This movie didn't quite live up to my high expectations, but overall I still really enjoyed it.",
]
candidate_labels = ["positive", "negative"]
hypothesis_template = "The sentiment of this review is {}."

classifier(sequences, candidate_labels, hypothesis_template=hypothesis_template)

By providing a more precise hypothesis template, we are able to see a more accurate classification of the second review.

> Note that sentiment classification is used here just as an illustrative example. The [Hugging Face Model Hub](https://huggingface.co/models?filter=text-classification) has a number of models trained specifically on sentiment tasks which can be used instead.

### Your turn: tune the hypothesis template

The pipeline turns each label into a hypothesis with a template. A template that fits the task can sharpen the scores. Try it on a topic-classification example.

**Steps**
1. Pick a sequence (for example a news headline) and 3 or 4 candidate labels.
2. Classify it with the default template, then again with a custom `hypothesis_template` such as `"This news article is about {}."`.
3. Compare the scores from both calls.

**Hint:** pass `hypothesis_template=` as a keyword argument to `classifier(...)`, and reuse `get_predictions_score(...)` to display each result.

**Expected:** the custom, task-specific template usually gives a higher score to the correct label.

In [ ]:
sequence = "The central bank raised interest rates again this quarter."
candidate_labels = ["economics", "sports", "politics", "weather"]

# Classify with the default template
default_pred = classifier(sequence, candidate_labels)

# Classify again with a custom hypothesis template
custom_pred = classifier(
    sequence, candidate_labels, hypothesis_template="This news article is about {}."
)

# Display both results and compare the scores
display(get_predictions_score(default_pred))
display(get_predictions_score(custom_pred))

**Explanation**

The default template phrases every label as 'This example is {}.', which is generic. A template that matches the domain ('This news article is about {}.') gives the NLI model a more natural hypothesis, so the correct label usually gets a higher score. Tuning the template is a cheap way to improve zero-shot results without any training.


## Zero-shot classification in more than 100 languages


Interested in using the pipeline for languages other than English? There is a cross-lingual model on top of XLM RoBERTa which you can use by passing `model='joeddav/xlm-roberta-large-xnli'` when creating the pipeline: 

In [ ]:
classifier = pipeline(
    "zero-shot-classification", model="joeddav/xlm-roberta-large-xnli", device=0
)  # device=0 uses the first GPU; set device=-1 to run on CPU

You can use it with any combination of languages. For example, let's classify a Russian sentence with English candidate labels:

In [ ]:
sequence = "За кого вы голосуете в 2020 году?"  # translation: "Who are you voting for in 2020?"
candidate_labels = ["Europe", "public health", "politics"]

classifier(sequence, candidate_labels)

Now let's do the same but with the labels in French:



In [ ]:
sequence = "За кого вы голосуете в 2020 году?"  # translation: "Who are you voting for in 2020?"
candidate_labels = ["Europe", "santé publique", "politique"]

classifier(sequence, candidate_labels)

As we discussed in the last section, the default hypothesis template is the English, `This text is {}.`. If you are working strictly within one language, it may be worthwhile to translate this to the language you are working with:

In [ ]:
sequence = "¿A quién vas a votar en 2020?"
candidate_labels = ["Europa", "salud pública", "política"]
hypothesis_template = "Este ejemplo es {}."

classifier(sequence, candidate_labels, hypothesis_template=hypothesis_template)

The model is fine-tuned on XNLI which includes 15 languages: Arabic, Bulgarian, Chinese, English, French, German, Greek, Hindi, Russian, Spanish, Swahili, Thai, Turkish, Urdu, and Vietnamese. The base model is trained on 85 more, so the model will work to some degree for any of those in the XLM RoBERTa training corpus (see the full list in appendix A of the [XLM Roberata paper](https://arxiv.org/abs/1911.02116)).

See the [model page](https://huggingface.co/joeddav/xlm-roberta-large-xnli) for more.

## Other pipeline tasks

Zero-shot classification is only one of many tasks the `pipeline` function supports. The same interface gives you text generation, sentiment analysis, translation, summarisation, and more, each backed by a suitable pretrained model.

[Read here](https://huggingface.co/docs/transformers/main_classes/pipelines) about different models available from Huggingface pipeline.

### Text generation

In [ ]:
text_gen = pipeline("text-generation", model="gpt2")  # to utilize GPU

In [ ]:
prompt = "Data Science is"
text_gen(prompt, max_length=30, num_return_sequences=3)

You can play around with different starting sentence. You can change `max_length` argument if you want shorter or longer sentences.

### Sentiment analysis

The sentiment analysis example in the beginning of the notebook can also be done using a sentiment analysis pipeline

### Your turn: a dedicated sentiment-analysis pipeline

Zero-shot is flexible, but a model trained for the task is simpler and often better. Build a `sentiment-analysis` pipeline and score a couple of sentences.

**Steps**
1. Create `pipeline("sentiment-analysis")` (no candidate labels needed, the model is already trained for sentiment).
2. Run it on two sentences, one clearly positive and one clearly negative.
3. Look at the label and score returned for each.

**Hint:** the sentiment pipeline returns a list of dicts like `[{'label': 'POSITIVE', 'score': 0.99}]`. You do not pass `candidate_labels`.

**Expected:** the positive sentence scores POSITIVE with high confidence, the negative one NEGATIVE.

In [ ]:
# Create a sentiment-analysis pipeline
sentiment = pipeline("sentiment-analysis")

reviews = [
    "I absolutely loved this film, it was fantastic.",
    "A complete waste of time, terrible from start to finish.",
]
# Run the pipeline on the reviews, then print the results
results = sentiment(reviews)
print(results)

**Explanation**

A dedicated sentiment-analysis pipeline loads a model already fine-tuned for the task, so you pass only the text, no candidate labels. It returns a label (POSITIVE or NEGATIVE) and a confidence score for each input. When a task is common and well defined, a purpose-trained model is simpler and usually more accurate than steering a general zero-shot model.


## Optional: push to the Hugging Face Hub

You can create a Hugging Face account and an access token if you want to create or push content to a repository (for example when training a model or editing a model card).

- Create an account at https://huggingface.co/
- After logging in, go to **Settings -> Access Tokens**
- Create a new token and give it write permissions
- Run these commands:
    - `brew install huggingface-cli`
    - `huggingface-cli login` and paste the access token from Hugging Face
    - Do not add an access token for GitHub if it asks

Reference: https://huggingface.co/docs/hub/security-tokens

## Summary

In this notebook you:

- Applied a zero-shot pipeline to custom sentences with your own labels.
- Explained how natural language inference turns labels into hypotheses to enable zero-shot classification.
- Used a multilingual model to classify text across many languages.
- Compared other pipeline tasks, including text generation and sentiment analysis.

## References & Further Reading

- [**Hugging Face: Pipelines**](https://huggingface.co/docs/transformers/main_classes/pipelines): Reference for every pipeline task used in this notebook.
- [**Hugging Face: Zero-Shot Classification**](https://huggingface.co/tasks/zero-shot-classification): Background on the task and the models that support it.